# gsplat-challenge: Phase-0 bootstrap

This notebook installs deps, fetches challenge artifacts, runs the local test suite, and then produces the **first real rasterized frame** — the Phase-0 gate that tells us the architecture works.

**Prereqs:** the `gsplat-challenge` branch must be pushed to your fork or origin. Edit `REPO_URL` and `BRANCH` below.

**Runtime:** GPU (T4 is fine; A100 faster).


In [ ]:
REPO_URL = 'https://github.com/IanPTan/comma-vc-dev.git'   # update if you use a fork
BRANCH = 'gsplat-challenge'

import os
if not os.path.isdir('comma-vc-dev'):
    !git clone -b {BRANCH} {REPO_URL}
%cd comma-vc-dev

In [ ]:
# Deps.  Colab already has torch + CUDA; gsplat picks the right wheel.
!pip install -q gsplat timm einops segmentation_models_pytorch safetensors av Pillow

In [ ]:
# Fetch the challenge artifacts we can't ship (~130 MB).
!bash scripts/fetch_challenge_deps.sh

In [ ]:
# Verify GPU + gsplat.
import torch, gsplat
print('torch', torch.__version__, 'cuda?', torch.cuda.is_available())
if torch.cuda.is_available():
    print('  device:', torch.cuda.get_device_name(0))
print('gsplat', getattr(gsplat, '__version__', '?'))

In [ ]:
# Run the local test suite on GPU. Expect 26 passes.
!python -m tests.test_smoke
!python -m tests.test_codec
!python -m tests.test_archive_roundtrip
!python -m tests.test_real_losses

## The Phase-0 gate: render one frame

Instantiate the untrained scene and rasterize one frame from a fixed sensible camera. Expect: a coherent (though noisy / random-colored) image where you can see the structure of the four populations.

In [ ]:
!python -m scripts.first_render --out first_frame.png

In [ ]:
# Display the result.
from PIL import Image
from IPython.display import display
img = Image.open('first_frame.png')
print('size:', img.size, 'mode:', img.mode)
display(img)

## What to look for

- **Not solid black**: at least *something* got rasterized.
- **No crashes**: gsplat + our arg-passing convention works end-to-end.
- **Rough structure**: you should see 2 lateral bands (roadside), some scattered dots near the ground (road plane), some scattered dots overhead (sky). Colors will be uniform-black or uniform-mid because the untrained scene has all colors initialized to 0 in pre-sigmoid logit space (= 0.5 gray after sigmoid).

If solid black: rasterizer args wrong (viewmat convention, near/far planes, or gsplat API version mismatch).  
If crash: paste traceback into the next chat message.